In [4]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("/workspaces/SebastianBot")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import importlib
from pathlib import Path

import astroid

from cloud.helper.event_grid import EventGridModel


def extract_event_grid_models() -> list[str]:
    """Extract all EventGridModel subclasses and their environment variable names."""
    base_path = Path("/workspaces/SebastianBot/cloud/functions")
    base_name = EventGridModel.__name__
    env_names: list[str] = []

    def extract_inherited_classnames(
        node: astroid.nodes.ClassDef, base_name: str
    ) -> str | None:
        base_names = []
        for b in node.bases:
            if hasattr(b, "name"):
                base_names.append(b.name)
            elif hasattr(b, "value") and hasattr(b.value, "name"):
                base_names.append(b.value.name)
        if base_name in base_names:
            return node.name
        return None

    for file in base_path.rglob("*.py"):
        module = astroid.parse(file.read_text())
        for node in module.body:
            if not isinstance(node, astroid.nodes.ClassDef):
                continue

            if not (name := extract_inherited_classnames(node, base_name)):
                continue

            # Convert file path to module path
            relative_path = file.relative_to(Path("/workspaces/SebastianBot"))
            module_path = str(relative_path.with_suffix("")).replace("/", ".")

            # Dynamically import the module and get the class
            imported_module = importlib.import_module(module_path)
            class_type = getattr(imported_module, name)
            env_names.append(class_type.env_name())

    return env_names

In [6]:
assert len(extract_event_grid_models()) > 0, "No EventGridModel subclasses found."

In [7]:
print(extract_event_grid_models())

['CompleteTaskEventGrid', 'CreateCalendarEventEventGrid', 'CreateTaskEventGrid', 'DeleteCalendarEventEventGrid', 'ModifyMailLabelEventGrid', 'SendTelegramMessageEventGrid']
